In [21]:
import pandas as pd
import numpy as np
import altair as alt
from sklearn.linear_model import LinearRegression

In [22]:
# Define paths to the individual CSV files

csv_files_output = {
    '50': 'emission_data/vllm_input_tok_summary/vLLM_50_word_summary.csv',
    '100': 'emission_data/vllm_input_tok_summary/vLLM_100_word_summary.csv',
    '250': 'emission_data/vllm_input_tok_summary/vLLM_250_word_summary.csv', 
    '500': 'emission_data/vllm_input_tok_summary/vLLM_500_word_summary.csv',
    '1000': 'emission_data/vllm_input_tok_summary/vLLM_1000_word_summary.csv',
    '2500': 'emission_data/vllm_input_tok_summary/vLLM_2500_word_summary.csv',
    '5000': 'emission_data/vllm_input_tok_summary/vLLM_5000_word_summary.csv',
    '7500': 'emission_data/vllm_input_tok_summary/vLLM_7500_word_summary.csv',
    # '10000': 'emission_data/vllm_input_tok_summary/vLLM_10000_word_summary.csv',
    # '15000': 'emission_data/vllm_input_tok_summary/vLLM_15000_word_summary.csv',
}

# Read the emissions data
emissions_data = pd.read_csv('input_tok_summary_vllm.csv')

In [23]:
# Initialize lists to store metadata
total_time = []
time_per_prompt = []
tok_per_sec = []
parameters_output = []
num_examples_output = []
num_prompts_output = []
total_emissions_output = []
cpu_energy_output = []
gpu_energy_output = []
ram_energy_output = []
total_energy_output = []
total_output_tokens_output = []
total_input_tokens_output = []
avg_input_tokens_output = []
avg_output_tokens_output = []

In [24]:
# Read and extract metadata from each CSV file
for model, file in csv_files_output.items():
    data = pd.read_csv(file)
    time = data.loc[data['Metric'] == 'Total Time', 'Value'].values[0]
    time_p_prompt = data.loc[data['Metric'] == 'AVG. Time / Prompt', 'Value'].values[0] / 1000 #Time is in ms
    tok_p_sec = data.loc[data['Metric'] == 'AVG. Tokens / Second', 'Value'].values[0]
    prompts = data.loc[data['Metric'] == 'Total Prompts', 'Value'].values[0]
    output_tokens = data.loc[data['Metric'] == 'Total Output Tokens', 'Value'].values[0]
    input_tokens = data.loc[data['Metric'] == 'Total Input Tokens', 'Value'].values[0]
    avg_i_tok = data.loc[data['Metric'] == 'AVG. Input Tokens / Prompt', 'Value'].values[0]
    avg_o_tok = data.loc[data['Metric'] == 'AVG. Output Tokens / Prompt', 'Value'].values[0]
    total_time.append(float(time))
    time_per_prompt.append(float(time_p_prompt))
    tok_per_sec.append(float(tok_p_sec))
    parameters_output.append(8)
    num_examples_output.append(int(model))
    num_prompts_output.append(int(prompts))
    total_output_tokens_output.append(float(output_tokens))
    total_input_tokens_output.append(float(input_tokens))
    avg_input_tokens_output.append(float(avg_i_tok))
    avg_output_tokens_output.append(float(avg_o_tok))    

In [25]:
# Extract emissions data
for model in csv_files_output.keys():
    model_emissions = emissions_data[emissions_data['project_name'].str.contains("vLLM_" + model + "_word_summary")]
    total_emissions_output.append(model_emissions['emissions'].values[0])
    cpu_energy_output.append(model_emissions['cpu_energy'].values[0])
    gpu_energy_output.append(model_emissions['gpu_energy'].values[0])
    ram_energy_output.append(model_emissions['ram_energy'].values[0])
    total_energy_output.append(model_emissions['energy_consumed'].values[0])


In [26]:
print(avg_input_tokens_output)
print(avg_output_tokens_output)
print(total_output_tokens_output)
print(total_emissions_output)

[441.0, 748.0, 1282.0, 2220.0, 3885.0, 7759.0, 14922.0, 25457.0]
[96.998, 152.015, 217.437, 285.218, 353.563, 427.407, 490.242, 567.433]
[96998.0, 152015.0, 217437.0, 285218.0, 353563.0, 427407.0, 490242.0, 567433.0]
[0.0056505383670293, 0.0070889279248674, 0.011731848486881, 0.0193749162103813, 0.0324112581576919, 0.075691368856632, 0.1424516719035163, 0.2168158357426798]


In [27]:
# Prepare data for regression and visualization
total_time = np.array(total_time)
time_per_prompt = np.array(time_per_prompt)
tok_per_sec = np.array(tok_per_sec)
parameters_output = np.array(parameters_output)
num_examples_output = np.array(num_examples_output)
num_prompts_output = np.array(num_prompts_output)
total_output_tokens_output = np.array(total_output_tokens_output)
total_input_tokens_output = np.array(total_input_tokens_output)
avg_input_tokens_output = np.array(avg_input_tokens_output)
avg_output_tokens_output = np.array(avg_output_tokens_output)
total_emissions_output = np.array(total_emissions_output)
cpu_energy_output = np.array(cpu_energy_output)
gpu_energy_output = np.array(gpu_energy_output)
ram_energy_output = np.array(ram_energy_output)
total_energy_output = np.array(total_energy_output)

In [28]:
print(total_time)

[  74.64758801   92.15549588  149.13437462  244.16428804  416.35407543
  957.70009375 1774.5068748  2685.21677136]


In [29]:
idle_gpu_power = 28*4 # 28W per GPU, 4 GPUs

total_idle_gpu_energy = (idle_gpu_power/1000)*(total_time/3600) # Convert W into kw and s into h
idle_gpu_energy_per_thousand_prompts = total_idle_gpu_energy / total_output_tokens_output * 100_000 * 1000,

gpu_energy_without_idle = gpu_energy_output - total_idle_gpu_energy
gpu_energy_without_idle_per_thousand_prompts = gpu_energy_without_idle / total_output_tokens_output * 100_000 * 1000,

In [30]:
# Calculate emissions per 10,000 prompts
emissions_per_thousand_prompts = {
    'Total Emissions per 100.000 output tokens': total_emissions_output / total_output_tokens_output * 100_000 * 1000,
    'CPU Energy per 100.000 output tokens': cpu_energy_output / total_output_tokens_output * 100_000 * 1000,
    'GPU Energy per 100.000 output tokens': gpu_energy_output / total_output_tokens_output * 100_000 * 1000,
    'GPU Energy per 100.000 output tokens (without idle)': np.concatenate([gpu_energy_without_idle_per_thousand_prompts]).flatten(),
    'GPU Energy per 100.000 output tokens (idle)': np.concatenate([idle_gpu_energy_per_thousand_prompts]).flatten(),
    'RAM Energy per 100.000 output tokens': ram_energy_output / total_output_tokens_output * 100_000 * 1000,
    'Total Energy per 100.000 output tokens': total_energy_output / total_output_tokens_output * 100_000 * 1000,
}

In [31]:
print(f"GPU Energy per 100.000 output tokens: {emissions_per_thousand_prompts['GPU Energy per 100.000 output tokens']}")
print(f"Idle GPU Energy per 100.000 output tokens: {emissions_per_thousand_prompts['GPU Energy per 100.000 output tokens (idle)']}")
print(f"GPU Energy without idle per 100.000 output tokens: {emissions_per_thousand_prompts['GPU Energy per 100.000 output tokens (without idle)']}")

GPU Energy per 100.000 output tokens: [ 5.87541002  4.74046557  5.54305568  7.00709818  9.37221276 18.23454556
 30.13218544 39.72636048]
Idle GPU Energy per 100.000 output tokens: [ 2.39424463  1.88603748  2.13383007  2.66330396  3.66362937  6.97113384
 11.26114869 14.72245663]
GPU Energy without idle per 100.000 output tokens: [ 3.48116539  2.85442809  3.4092256   4.34379422  5.7085834  11.26341172
 18.87103675 25.00390385]


In [32]:
print(emissions_per_thousand_prompts)

{'Total Emissions per 100.000 output tokens': array([ 5.8254174 ,  4.66330818,  5.39551617,  6.79302015,  9.16703902,
       17.70943594, 29.05741897, 38.20994474]), 'CPU Energy per 100.000 output tokens': array([1.10871066, 0.87337647, 0.98801631, 1.23310586, 1.69618298,
       3.22737009, 5.21344117, 6.81584135]), 'GPU Energy per 100.000 output tokens': array([ 5.87541002,  4.74046557,  5.54305568,  7.00709818,  9.37221276,
       18.23454556, 30.13218544, 39.72636048]), 'GPU Energy per 100.000 output tokens (without idle)': array([ 3.48116539,  2.85442809,  3.4092256 ,  4.34379422,  5.7085834 ,
       11.26341172, 18.87103675, 25.00390385]), 'GPU Energy per 100.000 output tokens (idle)': array([ 2.39424463,  1.88603748,  2.13383007,  2.66330396,  3.66362937,
        6.97113384, 11.26114869, 14.72245663]), 'RAM Energy per 100.000 output tokens': array([ 1.77658696,  1.39919699,  1.58311735,  1.97565852,  2.71769774,
        5.1708868 ,  8.35313952, 10.92083165]), 'Total Energy per 10

In [33]:
print(parameters_output)

[8 8 8 8 8 8 8 8]


In [34]:
# Define the test types and model types
test_types = ['Input-tok-vllm']
model_types = ['llama3']

# Define the parameters
parameters = np.concatenate([parameters_output])
num_examples = np.concatenate([num_examples_output])
num_prompts = np.concatenate([num_prompts_output])
total_out_tok = np.concatenate([total_output_tokens_output])
total_in_tok = np.concatenate([total_input_tokens_output])
avg_out_tok = np.concatenate([avg_output_tokens_output])
avg_in_tok = np.concatenate([avg_input_tokens_output])

actual_emissions_per_100k_output_tokens = emissions_per_thousand_prompts['Total Emissions per 100.000 output tokens']

actual_cpu_energy_per_100k_output_tokens = emissions_per_thousand_prompts['CPU Energy per 100.000 output tokens']

actual_gpu_energy_per_100k_output_tokens = emissions_per_thousand_prompts['GPU Energy per 100.000 output tokens']

actual_ram_energy_per_100k_output_tokens = emissions_per_thousand_prompts['RAM Energy per 100.000 output tokens']

actual_total_energy_per_100k_output_tokens = emissions_per_thousand_prompts['Total Energy per 100.000 output tokens']

actual_idle_gpu_energy_per_100k_output_tokens = emissions_per_thousand_prompts['GPU Energy per 100.000 output tokens (idle)']

actual_non_idle_gpu_energy_per_100k_output_tokens = emissions_per_thousand_prompts['GPU Energy per 100.000 output tokens (without idle)']

In [35]:
# Repeat test types and model types for each data point
test_type_column = np.concatenate([
    np.repeat(test_types[0], len(parameters_output))
])

model_type_column = np.concatenate([
    np.repeat(model_types[0], len(parameters_output))
])

In [36]:
print(len(actual_emissions_per_100k_output_tokens))
print(len(actual_cpu_energy_per_100k_output_tokens))
print(len(actual_gpu_energy_per_100k_output_tokens))
print(len(actual_ram_energy_per_100k_output_tokens))
print(len(actual_total_energy_per_100k_output_tokens))
print(len(actual_idle_gpu_energy_per_100k_output_tokens))
print(len(actual_non_idle_gpu_energy_per_100k_output_tokens))
print(len(test_type_column))
print(len(model_type_column))


8
8
8
8
8
8
8
8
8


In [37]:
# Create the dataframe
df = pd.DataFrame({
    'test_type': test_type_column,
    'model_type': model_type_column,
    'parameters': parameters,
    'num_examples': num_examples,
    'num_prompts': num_prompts,
    'total_time': total_time,
    'time_per_prompt': time_per_prompt,
    'tok_per_sec': tok_per_sec,
    'total_out_tok': total_out_tok,
    'total_in_tok': total_in_tok,
    'avg_out_tok': avg_out_tok,
    'avg_in_tok': avg_in_tok,
    'actual_emissions_per_100k_output_tokens': actual_emissions_per_100k_output_tokens,
    'actual_total_energy_per_100k_output_tokens': actual_total_energy_per_100k_output_tokens,
    'actual_cpu_energy_per_100k_output_tokens': actual_cpu_energy_per_100k_output_tokens,
    'actual_gpu_energy_per_100k_output_tokens': actual_gpu_energy_per_100k_output_tokens,
    'actual_ram_energy_per_100k_output_tokens': actual_ram_energy_per_100k_output_tokens,
    'actual_idle_gpu_energy_per_100k_output_tokens': actual_idle_gpu_energy_per_100k_output_tokens,
    'actual_non_idle_gpu_energy_per_100k_output_tokens': actual_non_idle_gpu_energy_per_100k_output_tokens,
})

df

,test_type,model_type,parameters,num_examples,num_prompts,total_time,time_per_prompt,tok_per_sec,total_out_tok,total_in_tok,avg_out_tok,avg_in_tok,actual_emissions_per_100k_output_tokens,actual_total_energy_per_100k_output_tokens,actual_cpu_energy_per_100k_output_tokens,actual_gpu_energy_per_100k_output_tokens,actual_ram_energy_per_100k_output_tokens,actual_idle_gpu_energy_per_100k_output_tokens,actual_non_idle_gpu_energy_per_100k_output_tokens
0,Input-tok-vllm,llama3,8,50,10000,74.647588,0.074648,1299.412380,96998.0,441000.0,96.998,441.0,5.825417,8.760708,1.108711,5.875410,1.776587,2.394245,3.481165
1,Input-tok-vllm,llama3,8,100,10000,92.155496,0.092155,1649.548934,152015.0,748000.0,152.015,748.0,4.663308,7.013039,0.873376,4.740466,1.399197,1.886037,2.854428
2,Input-tok-vllm,llama3,8,250,10000,149.134375,0.149134,1457.993843,217437.0,1282000.0,217.437,1282.0,5.395516,8.114189,0.988016,5.543056,1.583117,2.133830,3.409226
3,Input-tok-vllm,llama3,8,500,10000,244.164288,0.244164,1168.139707,285218.0,2220000.0,285.218,2220.0,6.793020,10.215863,1.233106,7.007098,1.975659,2.663304,4.343794
4,Input-tok-vllm,llama3,8,1000,10000,416.354075,0.416354,849.188277,353563.0,3885000.0,353.563,3885.0,9.167039,13.786093,1.696183,9.372213,2.717698,3.663629,5.708583
5,Input-tok-vllm,llama3,8,2500,10000,957.700094,0.957700,446.284805,427407.0,7759000.0,427.407,7759.0,17.709436,26.632802,3.227370,18.234546,5.170887,6.971134,11.263412
6,Input-tok-vllm,llama3,8,5000,10000,1774.506875,1.774507,276.269428,490242.0,14922000.0,490.242,14922.0,29.057419,43.698766,5.213441,30.132185,8.353140,11.261149,18.871037
7,Input-tok-vllm,llama3,8,7500,10000,2685.216771,2.685217,211.317390,567433.0,25457000.0,567.433,25457.0,38.209945,57.463033,6.815841,39.726360,10.920832,14.722457,25.003904


In [38]:
df[['avg_in_tok', 'actual_total_energy_per_100k_output_tokens']]

,avg_in_tok,actual_total_energy_per_100k_output_tokens
0,441.0,8.760708
1,748.0,7.013039
2,1282.0,8.114189
3,2220.0,10.215863
4,3885.0,13.786093
5,7759.0,26.632802
6,14922.0,43.698766
7,25457.0,57.463033


In [39]:
# Define chart width and height
chart_width = 1000
chart_height = 700

x_title = 'Average Input Tokens per Prompt'
x_data = 'avg_in_tok'
test_type = 'Input-tok-vllm'

scatter = alt.Chart(df).mark_circle(size=100).encode(
    x=alt.X(x_data, title=x_title),
    y=alt.Y('actual_total_energy_per_100k_output_tokens', title='Energy Consumption per 100.000 output tokens (in Wh)'),
    tooltip=[
        alt.Tooltip('parameters', title='Parameters (billions)'),
        alt.Tooltip('actual_emissions_per_100k_output_tokens', title='Actual Emissions per 100.000 output tokens'),
        alt.Tooltip('avg_out_tok', title='Average Output Tokens per Prompt'),
        alt.Tooltip('avg_in_tok', title='Average Input Tokens per Prompt'),
        alt.Tooltip('num_examples', title='Number of Examples'),
        alt.Tooltip('num_prompts', title='Number of Prompts'),
        alt.Tooltip('model_type', title='Model Type'),
        alt.Tooltip('test_type', title='Test Type'),
        alt.Tooltip('test_type', title='Test Type'),
    ]
).properties(
    title=f'Energy per 100.000 Output Tokens',
    width=chart_width,
    height=chart_height
)

# Create line plots for predicted emissions
line = scatter.transform_regression(x_data, 'actual_total_energy_per_100k_output_tokens', method="linear").mark_line()

# Create a combined chart with overlays for all emission types
combined_chart = alt.layer(scatter+line).resolve_scale(
    x='independent'
).properties(
    title='Energy per 100.000 output tokens',
    width=chart_width,  # Adjusted width for combined chart
    height=chart_height  # Adjusted height for combined chart
)

combined_chart.show()

alt.LayerChart(...)

In [40]:
# Store the results in a CSV file
df.to_csv('../results/data/input_tok_summary_vllm.csv', index=False)